<a href="https://colab.research.google.com/github/acuasami/Protocolo-/blob/main/prueba_OSM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -qq
!apt-get install -y -qq libspatialindex-dev # a veces necesario para rtree/geopandas
!pip install --upgrade pip
!pip install osmnx pandas networkx geopandas shapely rtree matplotlib
!pip install contextily
!pip install geopy

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libspatialindex6:amd64.
(Reading database ... 126675 files and directories currently installed.)
Preparing to unpack .../libspatialindex6_1.9.3-2_amd64.deb ...
Unpacking libspatialindex6:amd64 (1.9.3-2) ...
Selecting previously unselected package libspatialindex-c6:amd64.
Preparing to unpack .../libspatialindex-c6_1.9.3-2_amd64.deb ...
Unpacking libspatialindex-c6:amd64 (1.9.3-2) ...
Selecting previously unselected package libspatialindex-dev:amd64.
Preparing to unpack .../libspatialindex-dev_1.9.3-2_amd64.deb ...
Unpacking libspatialindex-dev:amd64 (1.9.3-2) ...
Setting up libspatialindex6:amd64 (1.9.3-2) ...
Setting up libspatialindex-c6:amd64 (1.9.3-2) ...
Setting up libspatialindex-dev:amd64 (1.9.3-2) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) .

In [ ]:
from typing import Tuple, List, Dict
import os
import sys
import math
import random
import pandas as pd
import osmnx as ox
import networkx as nx
from shapely.geometry import Point
import matplotlib.pyplot as plt
import re

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')


Exportar los datos del CSV y definir la Latitud y Longitud

In [ ]:
#CSV_PATH = '/content/drive/MyDrive/Bedolla/albergues_comedores_completo.csv'  # ajusta si está en otra ruta
CSV_PATH = '/content/albergues_comedores_completo.csv'

def try_read_csv(path):
    # intentos con varias codificaciones comunes
    encs = ['utf-8', 'latin-1', 'cp1252']
    for e in encs:
        try:
            return pd.read_csv(path, encoding=e)
        except Exception as ex:
            last_err = ex
    raise last_err

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"No se encontró {CSV_PATH} en el directorio actual. Sube el archivo a Colab o monta Drive.")

# leer el CSV (prueba distintas codificaciones si es necesario)
df_raw = try_read_csv(CSV_PATH)
print("Encabezados detectados:", list(df_raw.columns))

# Normalizar nombres de columna (caso-insensible, quitar espacios y acentos simples)
def norm_col(c):
    c2 = re.sub(r'[^a-z0-9]', '', c.lower(), flags=re.IGNORECASE)
    return c2

col_map = {norm_col(c): c for c in df_raw.columns}

# posibles alias
name_aliases = ['nombre','name','org','organization','ong','institucion','institución']
type_aliases = ['tipo','type','categoria','category']
lat_aliases = ['latitud','latitude','lat','y']
lon_aliases = ['longitud','longitude','lon','lng','x']

def find_column(alias_list):
    for a in alias_list:
        if a in col_map:
            return col_map[a]
    return None

col_name = find_column(name_aliases)
col_type = find_column(type_aliases)
col_lat  = find_column(lat_aliases)
col_lon  = find_column(lon_aliases)

if col_lat is None or col_lon is None:
    raise ValueError("No se encontraron columnas de latitud/longitud en el CSV. Busca nombres como 'Latitud','Longitud','lat','lon'.")

# construir DataFrame normalizado
df = df_raw.copy()
# crear columnas normalizadas
df_norm = pd.DataFrame()
df_norm['name'] = df[col_name] if col_name is not None else ''
df_norm['type'] = df[col_type] if col_type is not None else ''
# convertir lat/lon a float (remover comas, espacios)
def to_float_series(s):
    s2 = s.astype(str).str.replace(',', '.').str.strip()
    return pd.to_numeric(s2, errors='coerce')

df_norm['lat'] = to_float_series(df[col_lat])
df_norm['lon'] = to_float_series(df[col_lon])

# eliminar filas sin coordenadas válidas
before = len(df_norm)
df_norm = df_norm.dropna(subset=['lat','lon']).reset_index(drop=True)
after = len(df_norm)

print(f"Filas leídas: {before}, filas con coordenadas válidas: {after}")

# inspección rápida
display(df_norm.head(10))
print(df_norm.dtypes)

# Construir lista de waypoints (tuplas) y mostrar los primeros 10
waypoints = [{'name': row['name'], 'type': row['type'], 'lat': float(row['lat']), 'lon': float(row['lon'])}
             for _, row in df_norm.iterrows()]

print(f"Waypoints construidos: {len(waypoints)}. Primeros 10 (si existen):")
for i, w in enumerate(waypoints[:10]):
    print(i+1, w)

Encabezados detectados: ['id_municipio', 'Nombre', 'Tipo', 'Latitud', 'Longitud']
Filas leídas: 120, filas con coordenadas válidas: 119


,name,type,lat,lon
0,Belén (Casa del Migrante Tapachula),Albergue,14.887187,-92.244152
1,Ejército de Salvación (Tapachula),Albergue,14.920261,-92.252423
2,Jesús El Buen Pastor,Albergue,14.876642,-92.307416
3,Albergue Infantil La Esperanza,Albergue,14.906704,-92.259212
4,Jesús Esperanza en el Camino,Albergue,16.740910,-93.119554
5,Una Ayuda Para Ti Mujer Migrante,Albergue,16.743256,-93.122320
6,San Martín de Porres,Albergue,16.737983,-92.646998
7,Jtatic Samuel Ruíz García,Albergue,17.545107,-91.994705
8,Hogar de la Misericordia,Albergue,16.226065,-93.903525
9,La 72,Albergue,17.461921,-91.430963


name     object
type     object
lat     float64
lon     float64
dtype: object
Waypoints construidos: 119. Primeros 10 (si existen):
1 {'name': 'Belén (Casa del Migrante Tapachula)', 'type': 'Albergue', 'lat': 14.887187, 'lon': -92.2441517}
2 {'name': 'Ejército de Salvación (Tapachula)', 'type': 'Albergue', 'lat': 14.9202606, 'lon': -92.2524233}
3 {'name': 'Jesús El Buen Pastor', 'type': 'Albergue', 'lat': 14.8766423, 'lon': -92.307416}
4 {'name': 'Albergue Infantil La Esperanza', 'type': 'Albergue', 'lat': 14.9067037, 'lon': -92.259212}
5 {'name': 'Jesús Esperanza en el Camino', 'type': 'Albergue', 'lat': 16.7409097, 'lon': -93.1195536}
6 {'name': 'Una Ayuda Para Ti Mujer Migrante', 'type': 'Albergue', 'lat': 16.7432562, 'lon': -93.1223196}
7 {'name': 'San Martín de Porres', 'type': 'Albergue', 'lat': 16.7379825, 'lon': -92.6469981}
8 {'name': 'Jtatic Samuel Ruíz García', 'type': 'Albergue', 'lat': 17.5451071, 'lon': -91.9947052}
9 {'name': 'Hogar de la Misericordia', 'type': 'Albergue

# **Red vial **

In [ ]:
print("Versión OSMnx:", ox.__version__)

# Configuración básica (para v1.x, que es la más común en Colab)
ox.settings.use_cache = True
ox.settings.log_console = False


Versión OSMnx: 2.0.6


Punto de Usuario

In [ ]:
# Punto de inicio de prueba
start = start_lat, start_lon = 19.325521, -99.167807 #cdmx

#24.497440, -107.145388 (Mazatlán)

Calcular la ONG mas cercana

In [ ]:
from geopy.distance import geodesic

def ong_mas_cercana(pos_actual, waypoints):
    """
    Retorna la ONG más cercana a la posición actual,
    sin importar su ubicación (norte, sur, etc.)
    y que no sea frontera.
    """
    # Filtrar solo ONGs (excluyendo fronteras)
    ongs = [
        w for w in waypoints
        if str(w['type']).strip().lower() != 'frontera'
    ]

    if not ongs:
        return None

    # Calcular la distancia geodésica de cada ONG respecto al usuario
    for o in ongs:
        o['distancia'] = geodesic(pos_actual, (o['lat'], o['lon'])).kilometers

    # Devolver la ONG más cercana
    return min(ongs, key=lambda x: x['distancia'])

siguiente = ong_mas_cercana(start, waypoints)

if siguiente:
    print(f"ONG más cercana: {siguiente['name']} ({siguiente['distancia']:.2f} km)")
else:
    print("No se encontró ninguna ONG.")


ONG más cercana: Casa Tochán (8.55 km)


Descargar la vialidad

In [ ]:
import osmnx as ox
from geopy.distance import geodesic

# Usuario y ONG más cercana
start_point = start  # start ya es (lat, lon)
dest_point = (siguiente['lat'], siguiente['lon'])  # 'siguiente' es la ONG más cercana

# Distancia geodésica
distance_km = geodesic(start_point, dest_point).km

# Definir buffer en metros (distancia + margen extra)
buffer_m = (distance_km + 2) * 1000  # +2 km de margen extra

# Descargar grafo solo dentro del buffer desde el punto de inicio
G = ox.graph_from_point(start_point, dist=buffer_m, network_type="drive")

print("Grafo descargado con", len(G.nodes), "nodos y", len(G.edges), "aristas.")

Grafo descargado con 60003 nodos y 135401 aristas.


Heuristica haversine

In [ ]:
import math

def haversine_heuristic(u, v, G):
    """
    Devuelve la distancia Haversine entre dos nodos de G.
    u, v: nodos
    G: grafo OSMnx con atributos 'y' (lat) y 'x' (lon)
    """
    lat1, lon1 = G.nodes[u]['y'], G.nodes[u]['x']
    lat2, lon2 = G.nodes[v]['y'], G.nodes[v]['x']
    R = 6371000  # Radio de la Tierra en metros
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c


Encontrar nodos más cercanos al usuario y ONG

In [ ]:
# Nodo más cercano al usuario
orig_node = ox.distance.nearest_nodes(G, start_lon, start_lat)  # (lon, lat)

# Nodo más cercano a la ONG
dest_node = ox.distance.nearest_nodes(G, dest_point[1], dest_point[0])


A* usando la heurística Haversine

In [ ]:
import networkx as nx

route = nx.astar_path(
    G,
    orig_node,
    dest_node,
    heuristic=lambda u, v: haversine_heuristic(u, v, G),
    weight='length'
)

print("Ruta calculada con", len(route), "nodos.")


Ruta calculada con 137 nodos.


In [ ]:
import tracemalloc
import networkx as nx

tracemalloc.start()  # iniciar seguimiento de memoria

route = nx.astar_path(
    G,
    orig_node,
    dest_node,
    heuristic=lambda u, v: haversine_heuristic(u, v, G),
    weight='length'
)

current, peak = tracemalloc.get_traced_memory()
print(f"Memoria usada durante el cálculo: {current / 1024:.2f} KB (actual), {peak / 1024:.2f} KB (pico)")

tracemalloc.stop()


Memoria usada durante el cálculo: 5.91 KB (actual), 1043.53 KB (pico)


Dibujar el grafo y la ruta

In [ ]:
import folium

# --- 1️⃣ Crear el mapa centrado en el usuario ---
m = folium.Map(location=start, zoom_start=12, tiles="CartoDB positron")

# --- 2️⃣ Dibujar la ruta (ya calculada con A*) en morado ---
route_coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in route]
folium.PolyLine(
    route_coords,
    color="purple",
    weight=5,
    opacity=0.8,
    tooltip="Ruta óptima (A*)"
).add_to(m)

# --- 3️⃣ Marcar el punto de inicio (usuario) ---
folium.Marker(
    location=start,
    popup=folium.Popup("<b>Inicio</b><br>Usuario", max_width=250),
    tooltip="Inicio",
    icon=folium.Icon(color="blue", icon="user")
).add_to(m)

# --- 4️⃣ Marcar la ONG más cercana con icono diferente y morado ---
folium.Marker(
    location=dest_point,
    popup=folium.Popup(
        f"<div style='font-size:14px; font-weight:bold;'>{siguiente['name']}</div>"
        f"<div>Tipo: {siguiente['type']}</div>",
        max_width=300
    ),
    tooltip="ONG más cercana",
    icon=folium.Icon(color="purple", icon="star")  # icono cambiado y color morado
).add_to(m)

# --- 5️⃣ Marcar todas las ONGs y fronteras ---
for ong in waypoints:
    ong_type = str(ong['type']).strip().lower()

    # Determinar color según tipo
    if ong_type == 'frontera':
        color = 'green'
    elif ong['name'] == siguiente['name']:
        continue  # ya marcada como ONG más cercana
    else:
        color = 'gray'

    folium.Marker(
        location=(ong['lat'], ong['lon']),
        popup=folium.Popup(
            f"<div style='font-size:12px; font-weight:bold;'>{ong['name']}</div>"
            f"<div>Tipo: {ong['type']}</div>",
            max_width=250
        ),
        tooltip=ong['type'].capitalize(),
        icon=folium.Icon(color=color, icon="info-sign")
    ).add_to(m)

# --- 6️⃣ Mostrar el mapa ---
m


Buscar la siguiente ONG hacia el norte

In [ ]:
from geopy.distance import geodesic

def find_sorted_ongs(start, df, min_lat=None):
    candidates = []

    for _, row in df.iterrows():
        if str(row['type']).strip().lower() != 'frontera':
            # Filtrar por latitud si se indica
            if min_lat is None or row["lat"] > min_lat:
                ong_point = (row["lat"], row["lon"])
                dist = geodesic(start, ong_point).km
                candidates.append({
                    "name": row["name"],
                    "lat": row["lat"],
                    "lon": row["lon"],
                    "type": row["type"],
                    "distancia": dist
                })

    # Ordenar por distancia
    candidates.sort(key=lambda x: x['distancia'])
    return candidates


In [ ]:
# Obtener ONGs ordenadas por distancia desde el usuario
ongs_ordenadas = find_sorted_ongs(start, df_norm)  # df_norm es tu DataFrame o ajusta a waypoints si quieres

if len(ongs_ordenadas) == 0:
    print("No hay ONGs disponibles. Dirígete a la frontera.")
elif len(ongs_ordenadas) == 1:
    ong = ongs_ordenadas[0]
    print(f"Siguiente ONG: {ong['name']} ({ong['distancia']:.1f} km) - Tipo: {ong['type']}")
else:
    # Elegir la segunda ONG más cercana
    segunda = ongs_ordenadas[1]
    print(f"Siguiente recomendación: {segunda['name']} ({segunda['distancia']:.1f} km) - Tipo: {segunda['type']}")

Siguiente recomendación: Casa de los Amigos (12.5 km) - Tipo: Albergue
